### Data Cleaning & Preprocessing 

Dans ce notebook, nous allons :
1. Charger le dataset.
2. Appliquer les fonctions de nettoyage définies dans `src/data_cleaning.py`.
3. Préparer les nouvelles variables pour le funnel (conversion, intérêt, étapes).
4. Sauvegarder les données nettoyées dans `data/processed/`.

In [1]:
# Importations
import sys
import os
import pandas as pd
from pathlib import Path

src_path = Path.cwd().parent / "src"
sys.path.append(str(src_path))

from utils import setup_logger, ensure_directory_exists
from data_loading import load_data
import data_cleaning

# configuration du logger
logger = setup_logger("../reports/logs/data_cleaning.log")

## 1. Chargement des données brutes

In [2]:
# Chemins des fichiers
INPUT_PATH = str(Path.cwd().parent / "data/raw/bank-marketing.csv")
OUTPUT_PATH = str(Path.cwd().parent / "data/processed/bank-marketing_cleaned.csv")

#loading of data
df = load_data(INPUT_PATH)

#Appercu
print("Appercu des données: \n")
df.head()

2026-04-11 10:27:52,472 - INFO - |==> Data loaded successfully from e:\Certifs\Future_Interns\FUTURE_DS_03\data\raw\bank-marketing.csv


Appercu des données: 



,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,30,unemployed,married,primary,no,1787,no,no,cellular,19,oct,79,1,-1,0,unknown,no
1,33,services,married,secondary,no,4789,yes,yes,cellular,11,may,220,1,339,4,failure,no
2,35,management,single,tertiary,no,1350,yes,no,cellular,16,apr,185,1,330,1,failure,no
3,30,management,married,tertiary,no,1476,yes,yes,unknown,3,jun,199,4,-1,0,unknown,no
4,59,blue-collar,married,secondary,no,0,yes,no,unknown,5,may,226,1,-1,0,unknown,no


In [3]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4521 entries, 0 to 4520
Data columns (total 17 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   age        4521 non-null   int64 
 1   job        4521 non-null   object
 2   marital    4521 non-null   object
 3   education  4521 non-null   object
 4   default    4521 non-null   object
 5   balance    4521 non-null   int64 
 6   housing    4521 non-null   object
 7   loan       4521 non-null   object
 8   contact    4521 non-null   object
 9   day        4521 non-null   int64 
 10  month      4521 non-null   object
 11  duration   4521 non-null   int64 
 12  campaign   4521 non-null   int64 
 13  pdays      4521 non-null   int64 
 14  previous   4521 non-null   int64 
 15  poutcome   4521 non-null   object
 16  y          4521 non-null   object
dtypes: int64(7), object(10)
memory usage: 600.6+ KB
None


In [4]:
print(df.describe())

               age       balance          day     duration     campaign  \
count  4521.000000   4521.000000  4521.000000  4521.000000  4521.000000   
mean     41.170095   1422.657819    15.915284   263.961292     2.793630   
std      10.576211   3009.638142     8.247667   259.856633     3.109807   
min      19.000000  -3313.000000     1.000000     4.000000     1.000000   
25%      33.000000     69.000000     9.000000   104.000000     1.000000   
50%      39.000000    444.000000    16.000000   185.000000     2.000000   
75%      49.000000   1480.000000    21.000000   329.000000     3.000000   
max      87.000000  71188.000000    31.000000  3025.000000    50.000000   

             pdays     previous  
count  4521.000000  4521.000000  
mean     39.766645     0.542579  
std     100.121124     1.693562  
min      -1.000000     0.000000  
25%      -1.000000     0.000000  
50%      -1.000000     0.000000  
75%      -1.000000     0.000000  
max     871.000000    25.000000  


In [5]:
print("\n===Missing values===")
print(df.isnull().sum())


===Missing values===
age          0
job          0
marital      0
education    0
default      0
balance      0
housing      0
loan         0
contact      0
day          0
month        0
duration     0
campaign     0
pdays        0
previous     0
poutcome     0
y            0
dtype: int64


In [6]:
print("\nDuplicate rows:", df.duplicated().sum())


Duplicate rows: 0


## 2. Exécution du Pipeline de Nettoyage

In [7]:
# Data cleaning
df = data_cleaning.clean_unknown_values(df)
df = data_cleaning.clean_missing_values(df)
df = data_cleaning.clean_duplicates(df)
df = data_cleaning.clean_numeric_types(df)
df = data_cleaning.convert_target_to_binary(df)
df = data_cleaning.add_showed_interest(df)
df = data_cleaning.add_funnel_stage(df)


print("\nDimensions après nettoyage:")
df.head(10)

2026-04-11 10:28:04,572 - INFO - |==> Nettoyage des valeurs 'unknown'...
2026-04-11 10:28:04,579 - INFO - |==> Suppression des valeurs manquantes...
2026-04-11 10:28:04,585 - INFO -  3930 miss-values drop
2026-04-11 10:28:04,586 - INFO - |==> Suppression des lignes dupliquees...
2026-04-11 10:28:04,588 - INFO -  0 duplicates drop
2026-04-11 10:28:04,590 - INFO - |==> Nettoyage des types numeriques...
2026-04-11 10:28:04,593 - INFO - |==> Conversion of target in binary...
2026-04-11 10:28:04,594 - INFO - |==> Ajout de l'indicateur d'interet...
2026-04-11 10:28:04,597 - INFO - |==> Ajout de la colonne 'funnel_stage'...

Dimensions après nettoyage:


,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y,converted,showed_interest,funnel_stage
1,33,services,married,secondary,no,4789,yes,yes,cellular,11,may,220,1,339,4,failure,no,0,1,Highly_Engaged
2,35,management,single,tertiary,no,1350,yes,no,cellular,16,apr,185,1,330,1,failure,no,0,1,Highly_Engaged
5,35,management,single,tertiary,no,747,no,no,cellular,23,feb,141,2,176,3,failure,no,0,1,Engaged
6,36,self-employed,married,tertiary,no,307,yes,no,cellular,14,may,341,1,330,2,other,no,0,1,Highly_Engaged
9,43,services,married,primary,no,-88,yes,yes,cellular,17,apr,313,1,147,2,failure,no,0,1,Highly_Engaged
14,31,blue-collar,married,secondary,no,360,yes,yes,cellular,29,jan,89,1,241,1,failure,no,0,1,Engaged
17,37,admin.,single,tertiary,no,2317,yes,no,cellular,20,apr,114,1,152,2,failure,no,0,1,Engaged
19,31,services,married,secondary,no,132,no,no,cellular,7,jul,148,1,152,1,other,no,0,1,Engaged
38,33,management,married,secondary,no,3935,yes,no,cellular,6,may,765,1,342,2,failure,yes,1,1,Highly_Engaged
40,38,management,single,tertiary,no,11971,yes,no,unknown,17,nov,609,2,101,3,failure,no,0,1,Highly_Engaged


## 3. Sauvegarde des Données Propres

In [9]:
ensure_directory_exists(str(Path(OUTPUT_PATH).parent))
df.to_csv(OUTPUT_PATH, index=False, sep=';')  
logger.info(f"✅ Data saved in: {OUTPUT_PATH} ")
print(f"\nDimensions finales : {df.shape}")


2026-04-11 10:31:04,018 - INFO - ✅ Data saved in: e:\Certifs\Future_Interns\FUTURE_DS_03\data\processed\bank-marketing_cleaned.csv 

Dimensions finales : (775, 20)
